# Transformer Review, Part 1: From Words to Contextual Vectors

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/parisar/BME6938-Advanced-Biomedical-AI-Spring-2027/blob/main/lectures/notebooks/week02_transformer_review_part1.ipynb)

*Click the badge to open this notebook in Google Colab, then choose File > Save a copy in Drive so your changes are kept.*

**BME 6938 Advanced Biomedical AI · Week 2 · Reference notebook (ungraded)**

This notebook follows Lecture 2-1 step by step. Each section matches a part of the slides, uses the same running example
(*"The patient has a cold"*), and reproduces the same numbers, so you can see every idea from the lecture as working code.

| Section | Lecture step | What you will see |
|---|---|---|
| 1 | Tokens | How a tokenizer splits clinical text into pieces with integer IDs |
| 2 | Embeddings | The embedding table, and the 5 × d matrix for our sentence |
| 3 | The context problem | Why one vector per word is not enough, and how a transformer fixes it |
| 4 | Attention, step by step | Weighted average, dot product, softmax, with the slide numbers |
| 5 | Queries, keys, values | Attention for one token, then the matrix formula |
| 6 | Attention maps | A real attention map from BERT |
| 7 | Multi-head attention | Several heads in parallel |
| 8 | Word order | Why attention needs positions: learned and sinusoidal position vectors, RoPE, ALiBi |
| 9 | The encoder block | Attention, add & normalize, feed-forward, stacked |
| 10 | Using the output | A prediction head on top of the encoder |

**How to run it**
- **Google Colab:** *Runtime > Run all*. A GPU is not needed.
- **HiPerGator:** open it in Jupyter (Open OnDemand) with a PyTorch kernel. It runs on CPU in a few minutes.
- The first run downloads one pretrained model (BERT-base, about 440 MB).

**How to read it:** read the text above each cell, predict what the cell will print, then run it.
Nothing here is graded; the "Try it" boxes are optional.

## 0. Setup

In [ ]:
# Install the two libraries this notebook needs if they are missing (Colab already has them).
import importlib, subprocess, sys
for package in ["torch", "transformers"]:
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)          # fixed seed so random numbers are the same every run
np.set_printoptions(precision=2, suppress=True)
print("PyTorch", torch.__version__)

## 1. Step 1: split the text into tokens  *(slides: "Step 1: Split the Text into Tokens")*

A neural network computes with numbers, so the first step turns text into a list of **tokens**, each with an integer **ID**
(its row number in the vocabulary). We use the tokenizer of **BERT-base**, the model behind Thursday's paper.

In [ ]:
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Vocabulary size:", tokenizer.vocab_size)

sentence = "The patient has a cold"
tokens = tokenizer.tokenize(sentence)                 # text -> list of token strings
token_ids = tokenizer.convert_tokens_to_ids(tokens)   # token strings -> integer IDs

for t, i in zip(tokens, token_ids):
    print(f"{t:>10s}  ->  {i}")

Rare words are split into **word pieces** that the tokenizer has seen before. Pieces that continue a word start with `##`.
This is why the model never meets an "unknown" word, and also why medical terms often cost several tokens.

In [ ]:
for word in ["tachycardia", "hyperkalemia", "electrocardiogram", "fever"]:
    print(f"{word:>18s}  ->  {tokenizer.tokenize(word)}")

> **Try it:** tokenize a sentence from a clinical note you know (no real patient data). Which words are split the most?

## 2. Step 2: look up a vector for each token  *(slides: "Step 2: Look Up a Vector for Each Token")*

The **embedding table** is a learned matrix with one row per vocabulary entry and *d* columns. Looking up a token means
taking its row. For BERT-base, *d* = 768. Our 5 tokens become a **5 × 768 matrix**, one row per token.

In [ ]:
model = AutoModel.from_pretrained(MODEL_NAME, attn_implementation="eager")  # "eager" lets us read attention weights later
model.eval()                                       # inference mode: no dropout

embedding_table = model.embeddings.word_embeddings.weight   # shape: (vocabulary size, d)
print("Embedding table:", tuple(embedding_table.shape))

X = embedding_table[token_ids]                     # take one row per token
print("Sentence matrix X:", tuple(X.shape), "(tokens x d)")
print("First 4 numbers of each token vector:")
for t, row in zip(tokens, X):
    print(f"{t:>10s}", row[:4].detach().numpy())

## 3. The problem: context  *(slides: "One Word, Two Meanings" and "The Fix")*

The lookup gives a word the **same vector in every sentence**. In *"The patient has a cold"* the word means an infection; in
*"The patient feels cold"* it means a temperature. Below we compare the vector for "cold" in both sentences:

- **before** the transformer (the embedding lookup), and
- **after** the transformer (the final output of BERT, which is *contextual*).

The helper function returns the vector of one word in a sentence, from either place.

In [ ]:
def word_vector(sentence, word, contextual=True):
    # Return the vector for `word` in `sentence`: from BERT's output (contextual=True) or from the lookup table (False).
    inputs = tokenizer(sentence, return_tensors="pt")            # adds [CLS] at the start and [SEP] at the end
    position = inputs.input_ids[0].tolist().index(tokenizer.convert_tokens_to_ids(word))
    if not contextual:
        return embedding_table[inputs.input_ids[0, position]].detach()
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[0, position]                # one vector of size d per token

for contextual in [False, True]:
    v1 = word_vector("The patient has a cold", "cold", contextual)
    v2 = word_vector("The patient feels cold", "cold", contextual)
    similarity = F.cosine_similarity(v1, v2, dim=0).item()
    label = "after BERT (contextual)" if contextual else "lookup table (static)  "
    print(f"{label}: cosine similarity of 'cold' in the two sentences = {similarity:.3f}")

The static vectors are identical (similarity 1.000). After BERT the two vectors differ, because each "cold" has gathered
information from its neighbors. The rest of this notebook builds the mechanism that does this: **attention**.

## 4. Attention, step by step  *(slides: "Attention Is a Weighted Average", "Similarity", "Softmax")*

We now leave BERT and use small numbers so every step is visible. These are exactly the numbers on the slides.

### 4.1 A weighted average
The new vector for "cold" is a weighted average of all five token vectors. The weights are positive and sum to 1.

In [ ]:
words   = ["The", "patient", "has", "a", "cold"]
weights = np.array([0.05, 0.10, 0.30, 0.15, 0.40])      # how much "cold" borrows from each word (slide values)
print("Weights sum to", weights.sum())

d = 4                                                    # tiny vectors so we can print them
rng = np.random.default_rng(0)
vectors = rng.normal(size=(5, d))                        # stand-ins for the five token vectors

new_cold = (weights[:, None] * vectors).sum(axis=0)      # sum_i weight_i * vector_i
print("New vector for 'cold':", new_cold)

### 4.2 Where the weights come from: the dot product
The dot product multiplies matching entries and adds them up. It is large when two vectors point the same way.

In [ ]:
cold = np.array([1, 0, 2])
has  = np.array([2, 1, 1])
the  = np.array([-1, 2, 0])

print("cold . has =", cold @ has, "  (similar direction: high score)")
print("cold . The =", cold @ the, "  (different direction: low score)")

### 4.3 From scores to weights: softmax
Softmax turns any list of scores into positive weights that sum to 1:
$\text{weight}_i = e^{\text{score}_i} / \sum_j e^{\text{score}_j}$. These scores give the weights from 4.1.

In [ ]:
def softmax(scores):
    # Turn a vector of scores into positive weights that sum to 1.
    exp = np.exp(scores - scores.max())     # subtracting the max does not change the result but avoids overflow
    return exp / exp.sum()

scores = np.array([-0.5, 0.2, 1.3, 0.6, 1.6])
print("scores :", scores)
print("weights:", softmax(scores), " sum =", softmax(scores).sum())

## 5. Queries, keys, and values  *(slides: "One Vector, Three Jobs" to "The Matrix Form")*

Each token gets three vectors from three learned matrices: a **query** (what am I looking for?), a **key** (what do I offer
to be matched on?), and a **value** (what do I pass on?). Here the matrices are random; in a trained model they are learned.

### 5.1 Attention for one token

In [ ]:
d = 8
X = torch.randn(5, d)                      # 5 token vectors (stand-ins for embeddings + positions)
W_Q, W_K, W_V = (torch.randn(d, d) / math.sqrt(d) for _ in range(3))   # scaled so the scores stay moderate

Q, K, V = X @ W_Q, X @ W_K, X @ W_V        # every token gets a query, a key, and a value

q_cold = Q[4]                              # the query of "cold" (the 5th token)
scores = K @ q_cold / math.sqrt(d)         # (1) dot with every key, (2) divide by sqrt(d)
weights = torch.softmax(scores, dim=0)     # (3) softmax -> weights that sum to 1
new_cold = weights @ V                     # (4) weighted sum of the values

for w, a in zip(words, weights):
    print(f"{w:>8s}: weight {a:.2f}")
print("new 'cold' vector has shape", tuple(new_cold.shape))

### 5.2 All tokens at once: the matrix form
Stacking every query, key, and value into matrices computes attention for all tokens in one line:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Dividing by $\sqrt{d_k}$ keeps the dot products from growing with the vector size, which would make softmax too peaked.

In [ ]:
def attention(Q, K, V, mask=None):
    # Scaled dot-product attention. Q: (n_q, d), K and V: (n_k, d). Returns outputs (n_q, d) and weights (n_q, n_k).
    scores = Q @ K.T / math.sqrt(Q.shape[-1])          # every query against every key
    if mask is not None:
        scores = scores.masked_fill(mask, float("-inf"))   # used next week for the decoder
    weights = torch.softmax(scores, dim=-1)            # softmax along each row
    return weights @ V, weights

out, A = attention(Q, K, V)
print("output:", tuple(out.shape), "   weights:", tuple(A.shape))
print("row 'cold' of the weight matrix equals 5.1:", torch.allclose(A[4], weights))

# PyTorch has the same operation built in (and computes it efficiently on GPUs).
builtin = F.scaled_dot_product_attention(Q[None], K[None], V[None])[0]
print("matches torch.nn.functional.scaled_dot_product_attention:", torch.allclose(out, builtin, atol=1e-5))

## 6. Reading a real attention map  *(slides: "Reading an Attention Map")*

Now the real thing: the attention weights inside BERT for our sentence. Each row shows where one token looks; each row
sums to 1. BERT adds `[CLS]` and `[SEP]` tokens, and many heads put a lot of weight on them.

*Caution (as in the lecture):* one head in one layer is not an explanation of the model's prediction.

In [ ]:
inputs = tokenizer(sentence, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

print("layers:", len(outputs.attentions), "  shape per layer (batch, heads, tokens, tokens):", tuple(outputs.attentions[0].shape))

LAYER, HEAD = 5, 3                          # try other values
A = outputs.attentions[LAYER][0, HEAD].numpy()
labels = tokenizer.convert_ids_to_tokens(inputs.input_ids[0])

plt.figure(figsize=(5, 4.5))
plt.imshow(A, cmap="Oranges", vmin=0, vmax=1)
plt.xticks(range(len(labels)), labels, rotation=45); plt.yticks(range(len(labels)), labels)
plt.xlabel("keys (looked at)"); plt.ylabel("queries (looking)")
plt.title(f"BERT-base, layer {LAYER}, head {HEAD}"); plt.colorbar(); plt.tight_layout(); plt.show()

## 7. Multi-head attention  *(slides: "Multi-Head Attention")*

Several heads run in parallel, each with its own $W_Q, W_K, W_V$ on a smaller slice of the vector. Their outputs are
concatenated and mixed by one more matrix, $W_O$. BERT-base uses 12 heads of size 64 (12 × 64 = 768).

In [ ]:
class MultiHeadAttention(nn.Module):
    # Multi-head self-attention written out explicitly.
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads, self.d_head = n_heads, d_model // n_heads
        self.W_Q = nn.Linear(d_model, d_model)   # all heads' query projections in one matrix
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)   # mixes the concatenated heads

    def forward(self, x, mask=None):
        n = x.shape[0]                            # number of tokens
        def split(t):                             # (n, d_model) -> (heads, n, d_head)
            return t.view(n, self.n_heads, self.d_head).transpose(0, 1)
        Q, K, V = split(self.W_Q(x)), split(self.W_K(x)), split(self.W_V(x))
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_head)     # (heads, n, n)
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        heads = weights @ V                                            # (heads, n, d_head)
        concat = heads.transpose(0, 1).reshape(n, -1)                  # (n, d_model)
        return self.W_O(concat), weights

mha = MultiHeadAttention(d_model=768, n_heads=12)
x = torch.randn(5, 768)
y, w = mha(x)
print("input", tuple(x.shape), "-> output", tuple(y.shape), "| attention weights per head", tuple(w.shape))

## 8. Word order  *(slides: "Attention Ignores Word Order" to "Relative Positions"; clips 3 and 4)*

### 8.1 Attention alone cannot see order
Word order matters: *"chest pain"* and *"pain chest"* are not the same. But if we swap the two input tokens,
attention gives each token the same output vector as before, only swapped. On its own it cannot tell the two orders apart.

In [ ]:
x = torch.randn(2, 768)                         # two token vectors: "chest", "pain" (random stand-ins)
swap = torch.tensor([1, 0])                     # the swapped order: "pain", "chest"
y_original, _ = mha(x)                          # attention outputs for "chest pain"
y_swapped, _ = mha(x[swap])                     # attention outputs for "pain chest"
print("same output vectors, only swapped:", torch.allclose(y_original[swap], y_swapped, atol=1e-5))

### 8.2 The fix: add a position vector to each token vector
Each position gets its own vector, and it is added to the token vector: **token vector + position vector = input vector**.
The input vector now carries both the word and where it sits. There are two common ways to get the position vectors.

### 8.3 Option 1: learn one vector per position (BERT)
BERT learns a table with one row per position, exactly like the embedding table for words. *"The patient has a cold"*
uses rows 1 to 5. The table has 512 rows, so a longer input has no row to look up.

In [ ]:
position_table = model.embeddings.position_embeddings.weight          # BERT's learned position table
print("position table:", tuple(position_table.shape), "(rows = positions, columns = d)")
print("longest input BERT can read:", model.config.max_position_embeddings, "tokens")

### 8.4 Option 2: compute position vectors from waves (original transformer)
The original transformer computes the position vectors instead of learning them. Each pair of dimensions follows a sine and a
cosine wave, and each pair uses a different speed. To build the vector for one position, read the height of every wave at that
position.

The clip draws four dimensions: a sine and a cosine at a fast speed (1 per position), and a sine and a cosine at a slow speed
(0.25 per position, chosen so the change is visible). Reading them at position 1 gives the vector **0.84, 0.54, 0.25, 0.97**.

In [ ]:
speeds = [1.0, 0.25]                                         # fast and slow wave speeds used in the clip
for p in range(1, 6):                                        # positions 1 to 5
    vector = []                                              # the position vector for position p
    for w in speeds:                                         # each speed gives one sine and one cosine dimension
        vector += [np.sin(w * p), np.cos(w * p)]             # read the height of both waves at position p
    print(f"position {p}: {np.round(vector, 2)}")            # fast pair changes a lot, slow pair drifts slowly

Moving from one position to the next, the fast pair jumps while the slow pair barely moves, like the second hand and the hour
hand of a clock. Together they give every position its own pattern.

The full recipe from *Attention Is All You Need* uses $d/2$ speeds, from 1 down to $1/10000$. With positions added,
swapping the tokens no longer just swaps the outputs.

In [ ]:
def sinusoidal_positions(n_positions, d_model):
    # Position vectors from "Attention Is All You Need": even dimensions use sine, odd use cosine.
    position = torch.arange(n_positions)[:, None]                                  # (n, 1)
    frequency = torch.exp(-math.log(10000.0) * torch.arange(0, d_model, 2) / d_model)  # (d/2,) one speed per pair
    P = torch.zeros(n_positions, d_model)                                          # one row per position
    P[:, 0::2] = torch.sin(position * frequency)                                   # sine in even dimensions
    P[:, 1::2] = torch.cos(position * frequency)                                   # cosine in odd dimensions
    return P

P = sinusoidal_positions(50, 128)                                                  # 50 positions, 128 dimensions
plt.figure(figsize=(7, 3))                                                         # fast waves on the left, slow on the right
plt.imshow(P, aspect="auto", cmap="RdBu"); plt.colorbar()
plt.xlabel("dimension"); plt.ylabel("position"); plt.title("Sinusoidal position vectors"); plt.tight_layout(); plt.show()

y_a, _ = mha(x + sinusoidal_positions(2, 768))                 # "chest pain" with positions added
y_b, _ = mha(x[swap] + sinusoidal_positions(2, 768))           # "pain chest" with positions added
print("with positions, swapping no longer just swaps:", not torch.allclose(y_a[swap], y_b, atol=1e-5))

Both options are **absolute** positions: they are added once, at the input, and every later layer has to carry that
information forward. Newer models handle position inside attention instead.

### 8.5 Relative positions with RoPE  *(clip 4)*
An attention score is a dot product, and a dot product depends on the angle between the two vectors:
$q \cdot k = |q|\,|k|\cos(\text{angle})$. Smaller angle, higher score.

**Rotary position embeddings (RoPE)** turn each query and key by an angle set by its position, here 15° per position.
We use the clip's two-dimensional vectors: the query for *chest* at 20° and the key for *pain* at 35°, 15° apart, with score 1.04.

In [ ]:
def at_angle(length, degrees):
    # A 2-D vector with the given length, pointing at the given angle.
    a = np.radians(degrees)                                  # degrees to radians
    return length * np.array([np.cos(a), np.sin(a)])

def rotate(v, degrees):
    # Rotate a 2-D vector by the given angle (this is what RoPE does to each pair of dimensions).
    a = np.radians(degrees)
    R = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])   # 2-D rotation matrix
    return R @ v

q_chest = at_angle(1.2, 20)                                  # query for "chest", 20°
k_pain = at_angle(1.0767 / 1.2, 35)                          # key for "pain", 35°, so the angle between them is 15°
print("no position yet: score =", round(q_chest @ k_pain, 2))

STEP = 15                                                    # degrees of rotation per position
def rope_score(pos_chest, pos_pain):
    # Rotate the query and the key by their positions, then take the dot product.
    return rotate(q_chest, STEP * pos_chest) @ rotate(k_pain, STEP * pos_pain)

for pc, pp in [(2, 3), (7, 8), (2, 6)]:
    print(f"chest at {pc}, pain at {pp}: distance {pp - pc}, score = {rope_score(pc, pp):.2f}")

Positions 2 and 3 and positions 7 and 8 give the same score, 0.93, because both vectors turn but the angle between them
stays 30°. Only the distance matters. Four positions apart, the angle grows to 75° and the score drops to 0.28.

Real models rotate many pairs of dimensions, each at its own speed: the same waves as in section 8.4. The check below uses
random 64-dimensional vectors and confirms that the score depends only on the distance between the two positions.

In [ ]:
def rope(v, position, base=10000.0):
    # Apply RoPE to a vector of even length: rotate each pair of dimensions by position x its own speed.
    d = len(v)
    speeds = base ** (-np.arange(0, d, 2) / d)               # one speed per pair, fast to slow
    a = position * speeds                                    # rotation angle of each pair
    out = np.empty(d)                                        # rotated vector
    out[0::2] = v[0::2] * np.cos(a) - v[1::2] * np.sin(a)    # first coordinate of each rotated pair
    out[1::2] = v[0::2] * np.sin(a) + v[1::2] * np.cos(a)    # second coordinate of each rotated pair
    return out

rng = np.random.default_rng(0)                               # fixed random generator
q, k = rng.normal(size=64), rng.normal(size=64)              # one query and one key, 64 dimensions
for m, n in [(2, 3), (7, 8), (100, 101), (2, 6)]:
    print(f"query at {m:3d}, key at {n:3d} (distance {n - m}): score = {rope(q, m) @ rope(k, n):.4f}")

### 8.6 Relative positions with ALiBi  *(clip 4)*
**ALiBi** leaves the vectors alone. It subtracts a penalty from each attention score, proportional to the distance:
penalty = slope × distance. We use the scores for *cold* from section 4 and a slope of 0.25.

In [ ]:
words = ["The", "patient", "has", "a", "cold"]
scores = np.array([-0.5, 0.2, 1.3, 0.6, 1.6])              # scores for "cold" (same as the slides and clip 2)
distance = np.array([4, 3, 2, 1, 0])                       # how far each word is from "cold"
slope = 0.25                                               # ALiBi slope
penalty = -slope * distance + 0.0                         # penalty grows with distance (+ 0.0 prints -0 as 0)
new_scores = scores + penalty                              # subtract the penalty from each score

print("scores:     ", scores)
print("penalty:    ", penalty)
print("new scores: ", new_scores)
print()
before, after = softmax(scores), softmax(new_scores)       # softmax from section 4.3
for w, b, a in zip(words, before, after):
    print(f"{w:>8s}: weight {b:.2f} -> {a:.2f}")           # nearby words gain weight, distant words lose it

After ALiBi, *cold* puts more weight on itself (0.40 to 0.53) and less on the farthest word (0.05 to 0.02).

Today RoPE is used in nearly every open large language model (Llama, Qwen, DeepSeek, Gemma) and in newer encoders such
as ModernBERT. To read records longer than they were trained on, models stretch the rotation angles after pretraining.

> **Try it:** in the RoPE cell, change `STEP` to 5 or 30 degrees. Does the score still depend only on the distance?
> In the ALiBi cell, try a slope of 1.0. What happens to the weight on *cold*?

## 9. The encoder block  *(slides: "Block Step 1" to "Stack the Blocks")*

The block wraps attention with three small pieces:
1. **Multi-head self-attention** (section 7),
2. **Add & normalize**: add the input back (residual connection), then layer normalization,
3. **Feed-forward network**: the same two-layer network applied to every token (d → 4d → d), then add & normalize again.

Because the output has the same shape as the input, blocks can be stacked.

In [ ]:
class EncoderBlock(nn.Module):
    # One transformer encoder block (post-norm, as in the original paper and BERT).
    def __init__(self, d_model=768, n_heads=12, d_ff=3072):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.feed_forward = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attended, _ = self.attention(x)
        x = self.norm1(x + attended)                 # step 2: residual + normalize
        x = self.norm2(x + self.feed_forward(x))     # step 3: feed-forward, residual + normalize
        return x

encoder = nn.Sequential(*[EncoderBlock() for _ in range(12)])   # BERT-base stacks 12 blocks
x = torch.randn(5, 768)
print("input", tuple(x.shape), "-> output", tuple(encoder(x).shape), " (shape in = shape out)")

n_params = sum(p.numel() for p in encoder.parameters())
print(f"parameters in 12 blocks: {n_params / 1e6:.0f} M (BERT-base is 110 M in total, including embeddings)")

Where do the parameters live? Compare the attention and feed-forward parts of one block:

In [ ]:
block = EncoderBlock()
count = lambda module: sum(p.numel() for p in module.parameters())
print(f"attention:    {count(block.attention) / 1e6:.1f} M")
print(f"feed-forward: {count(block.feed_forward) / 1e6:.1f} M   <- most of the parameters")

## 10. Using the encoder output  *(slides: "Using the Encoder Output")*

The encoder gives one contextual vector per token. A small **prediction head** turns them into task outputs:
- a **sentence** label from the `[CLS]` vector (for example, is the finding negated?),
- a **token** label from each token's vector (for example, PROBLEM or MEDICATION).

The heads below are untrained, so their outputs are meaningless; fine-tuning trains them on labeled examples. This is the
recipe used by BERT and GatorTron (Thursday's papers).

In [ ]:
inputs = tokenizer("no fever on aspirin", return_tensors="pt")
with torch.no_grad():
    H = model(**inputs).last_hidden_state[0]          # (tokens, 768) contextual vectors

sentence_head = nn.Linear(768, 2)                      # 2 classes: negated / not negated
token_head = nn.Linear(768, 3)                         # 3 tags: O, PROBLEM, MEDICATION

print("tokens:", tokenizer.convert_ids_to_tokens(inputs.input_ids[0]))
print("sentence probabilities from [CLS]:", torch.softmax(sentence_head(H[0]), dim=-1).detach().numpy())
print("token tag probabilities, shape:", tuple(torch.softmax(token_head(H), dim=-1).shape))

## Recap

| Step | Code in this notebook |
|---|---|
| Tokenize | `tokenizer.tokenize`, `convert_tokens_to_ids` |
| Embed | `model.embeddings.word_embeddings.weight[token_ids]` |
| Attention | `attention(Q, K, V)` = softmax(QKᵀ/√d)V |
| Multi-head | `MultiHeadAttention` |
| Positions | `position_embeddings` (learned), `sinusoidal_positions`, `rope`, ALiBi penalty |
| Encoder block | `EncoderBlock`, stacked with `nn.Sequential` |
| Prediction head | `nn.Linear` on `[CLS]` or on every token |

**Next week (Part 2):** the causal mask, the decoder, next-token prediction with GPT-2, cross-attention, and images as tokens (ViT).

> **Try it (optional):** in section 6, find a layer and head where "cold" puts most of its weight on "has". In section 3,
> compare "discharge" in *"ready for discharge home"* and *"purulent discharge from the wound"*.